# Task 1 — GPT-style character-level LLM from scratch on TinyStories
**DATA266 Lab 1 · Member: Shibin Thomas**

This notebook is the walkthrough of my Task 1 work. The real logic lives in the tested modules
next to it in `src/` (`data_prep.py`, `model.py`, `train.py`, `evaluate.py`, `analyze_failures.py`);
the notebook calls them, so the notebook and the one-command CLI give identical results.

| Brief section | Where |
|---|---|
| 1.1 Data preprocessing (char tokenisation, fixed-length sequences, `char_to_idx`/`idx_to_char`, own 100K/10K split) | §1 below, `src/data_prep.py` |
| 1.2 GPT implementation (MHSA, LayerNorm, FFN, residuals, causal mask, token + positional embeddings, LM head) | §2, `src/model.py` |
| 1.3 Training (cross-entropy, warm-up + cosine schedule, ≥10 epochs, loss curves, greedy/temperature generation) | §3–4, `src/train.py`, `src/evaluate.py` |
| 1.4 Failure analysis (3 cases) | §5, `failure_analysis.md` |
| All Task 1 metrics | §4, `metrics_report.csv` |

**No prebuilt Transformer or attention modules are used** (no `nn.MultiheadAttention`, `nn.Transformer*`,
`nn.LayerNorm`, `F.scaled_dot_product_attention`) — enforced by `tests/test_model.py`.

> For the long 10-epoch GPU run, the CLI is recommended (it survives notebook disconnects and can resume):
> `python task1_llm/shibin_thomas/src/run_pipeline.py --config task1_llm/shibin_thomas/configs/gpt_char_v1.yaml`
> then execute this notebook: with `RUN_TRAINING = "auto"` it re-uses that finished run instead of training again.

In [ ]:
# ---- Settings -------------------------------------------------------------------------------
CONFIG = "task1_llm/shibin_thomas/configs/gpt_char_v1.yaml"   # smoke test: configs/smoke.yaml
OVERRIDES = []           # e.g. ["training.batch_size=32"]
RUN_TRAINING = "auto"    # "auto": train only if no finished run of this config exists; True / False to force
RUN_ID = "latest"        # run to analyse when not training: a run id under checkpoints/, or "latest"

In [ ]:
import sys, json, math
from pathlib import Path

# locate the repo root from wherever the notebook is opened (no hard-coded personal paths)
here = Path.cwd().resolve()
REPO = next((p for p in [here, *here.parents] if (p / "task1_llm").is_dir()), None)
assert REPO is not None, "open this notebook from inside the cloned repository"
SRC = REPO / "task1_llm" / "shibin_thomas" / "src"
sys.path.insert(0, str(SRC))

import numpy as np, torch, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

from utils import load_config, repo_path, read_json, get_device, run_dirs, latest_run_id, rel
from data_prep import prepare_data, load_vocab, encode, decode, CharStream, EOS_ID
from model import GPTCharLM, GPTConfig
cfg = load_config(CONFIG, OVERRIDES)
device = get_device(cfg.get("device", "auto"))
print("repo:", REPO.name, "| config:", cfg["_config_path"], "| device:", device, "| torch", torch.__version__)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## 1. Data preprocessing (1.1)
1. **Load + clean** the shared raw TinyStories file. Typographic unicode (’ “ ” — …) is mapped to ASCII,
   whitespace is normalised, and stories that still contain non-ASCII characters or are shorter than
   50 characters are dropped (counts are logged). This keeps the character vocabulary small (~95)
   and avoids wasting capacity on rare mojibake symbols.
2. **My own split**: from the clean pool I sample, with my own seed (`split_seed: 266`), **100,000 training
   and 10,000 validation stories** (disjoint). The raw-line indices are saved in `split_indices.npz`.
3. **Character vocabulary** `char_to_idx` / `idx_to_char` is built from the *training* split only.
   Two special ids: `0 = <eos>` (end of story — lets the model learn where stories end) and
   `1 = <unk>` (validation character never seen in training).
4. Every story is encoded, followed by `<eos>`, and concatenated into one integer stream per split (`uint8`).
5. **Fixed-length input–target sequences**: windows of `block_size = 256` characters,
   `x = s[i : i+256]`, `y = s[i+1 : i+257]` (target = input shifted by one character).

In [ ]:
processed = prepare_data(cfg)
meta = read_json(processed / "meta.json")
char_to_idx, idx_to_char = load_vocab(processed)
print(json.dumps({k: meta[k] for k in ["raw_status_counts", "vocab_size", "train", "val",
                                       "sequences_per_epoch_estimate", "optimizer_steps_per_epoch_estimate"]}, indent=1))

In [ ]:
# own split: sizes + disjointness check
split = np.load(processed / "split_indices.npz")
tr, va = split["train_lines"], split["val_lines"]
print(f"train stories={len(tr):,}  val stories={len(va):,}  overlap={len(np.intersect1d(tr, va))}")

# story length distribution (characters) of my training split, decoded back from the stream
train_ids = np.fromfile(processed / "train.bin", dtype=np.uint8)
eos_pos = np.flatnonzero(train_ids == EOS_ID)
lengths = np.diff(np.concatenate([[-1], eos_pos])) - 1
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(lengths, bins=80, color="#1f5aa6")
ax.axvline(cfg["model"]["block_size"], color="#c0392b", ls="--", label=f"block_size = {cfg['model']['block_size']}")
ax.set(xlabel="story length (characters)", ylabel="stories", title="Training-split story lengths"); ax.legend()
plt.show()
print(f"mean={lengths.mean():.0f}  median={np.median(lengths):.0f}  p95={np.percentile(lengths, 95):.0f} chars")

In [ ]:
# the char_to_idx / idx_to_char dictionaries
print("vocab size:", len(char_to_idx))
print("char_to_idx:", {repr(c): i for c, i in list(char_to_idx.items())})
print("idx_to_char[0..5]:", {i: idx_to_char[i] for i in range(6)})

text = "Once upon a time, Lily saw a cat."
ids = encode(text, char_to_idx)
print("\nencode:", ids)
print("decode:", decode(ids, idx_to_char))
assert decode(ids, idx_to_char) == text

In [ ]:
# fixed-length (input, target) pairs exactly as the training loop sees them
T = cfg["model"]["block_size"]
stream = CharStream(processed / "train.bin", T, torch.device("cpu"))
x, y = next(stream.batches(stream.eval_starts(4), batch_size=4))
print("x shape", tuple(x.shape), "| y shape", tuple(y.shape))
print("x[0][:12] =", x[0, :12].tolist())
print("y[0][:12] =", y[0, :12].tolist(), "  (shifted by one)")
print("\nx text:", repr(decode(x[0, :80].tolist(), idx_to_char)))
print("y text:", repr(decode(y[0, :80].tolist(), idx_to_char)))
assert torch.equal(x[0, 1:], y[0, :-1])

## 2. GPT model implemented from scratch (1.2)
Pre-LayerNorm GPT (GPT-2 layout), all components hand-written in `src/model.py`:

* `LayerNorm` — mean/variance normalisation with learnable γ, β (computed in fp32 for mixed-precision stability)
* `CausalSelfAttention` — fused Q/K/V projection, split into heads, `softmax(QKᵀ/√d_k + mask)·V`,
  heads concatenated, output projection; **causal mask** = lower-triangular boolean buffer, future
  positions are set to −∞ before the softmax
* `FeedForward` — Linear(d, 4d) → GELU → Linear(4d, d) → dropout
* `TransformerBlock` — `x = x + MHSA(LN(x)); x = x + FFN(LN(x))` (residual connections)
* learnable **token embedding** (`vocab × d`) + learnable **positional embedding** (`block_size × d`)
* final LayerNorm + **LM head** `Linear(d → vocab)` producing next-character logits (weight-tied to the token embedding)

In [ ]:
gcfg = GPTConfig(vocab_size=meta["vocab_size"], **cfg["model"])
model = GPTCharLM(gcfg)
print(model)
print(f"\nparameters: {model.num_params():,} total | {model.num_params(non_embedding=True):,} non-embedding")

In [ ]:
# Causal-masking check: perturbing characters at positions >= cut must not change logits before cut.
model.eval()
cut = T // 2
xx = x[:1, :T].clone()
xx2 = xx.clone(); xx2[0, cut:] = (xx2[0, cut:] + 5) % meta["vocab_size"]
with torch.no_grad():
    l1, _ = model(xx); l2, _ = model(xx2)
print(f"max |Δlogits| at positions <  {cut}:", (l1[0, :cut] - l2[0, :cut]).abs().max().item(), "(must be 0)")
print(f"max |Δlogits| at positions >= {cut}:", (l1[0, cut:] - l2[0, cut:]).abs().max().item())
print("initial loss (should be ≈ ln V =", round(math.log(meta['vocab_size']), 3), "):",
      round(model(x[:, :T], y[:, :T])[1].item(), 3))

## 3. Training (1.3)
* **Loss**: cross-entropy over next-character predictions (mean over all positions).
* **Optimiser**: AdamW (β = 0.9/0.95, weight decay 0.1 on matrices/embeddings only), gradient clipping at 1.0.
* **LR schedule**: linear **warm-up** over the first 2 % of steps to 6e-4, then **cosine decay** to 6e-5.
* **10 epochs**; one epoch = every training character is a target once (non-overlapping 256-char windows,
  random offset + shuffle each epoch).
* Mixed precision (bf16/fp16) on GPU; after each epoch: full validation pass, eval-mode train loss,
  checkpointing (best + resumable last), and a greedy sample in the raw log.
* Raw, append-only logs: `reproducibility/raw_logs/task1_llm/shibin_thomas/<run_id>/`.

In [ ]:
from train import train
from utils import MEMBER_DIR
finished = sorted((p.parent.name for p in (MEMBER_DIR / "outputs").glob(f"{cfg['run_name']}_*/train_summary.json")))
if RUN_TRAINING == "auto":
    RUN_TRAINING = not finished
if RUN_TRAINING:
    RUN_ID = train(cfg)
elif RUN_ID == "latest":
    RUN_ID = finished[-1]          # newest finished run (run ids end in a sortable timestamp)
print("trained now" if RUN_TRAINING else "re-using finished run")
dirs = run_dirs(RUN_ID)
print("run:", RUN_ID)
epochs_df = pd.read_csv(dirs["raw_logs"] / "epochs.csv")
epochs_df

## 4. Evaluation — loss curves, generation, all Task 1 metrics
`evaluate.py` re-loads the **best** checkpoint (lowest validation loss) and computes every metric in the brief,
using the team's shared metric definitions in `task1_llm/shared_eval/metrics.py` and the shared prompts in
`task1_llm/shared_eval/prompts.json`. Generation uses **greedy decoding** and **temperature sampling (T = 0.8)**.

In [ ]:
from evaluate import evaluate
metrics = evaluate(cfg, RUN_ID)
report = pd.read_csv(dirs["outputs"] / "metrics_report.csv")
report

In [ ]:
display(Image(filename=str(dirs["outputs"] / "loss_curves.png")))
display(Image(filename=str(dirs["outputs"] / "lr_schedule.png")))
display(Image(filename=str(dirs["outputs"] / "grad_norm.png")))

In [ ]:
samples = read_json(dirs["outputs"] / "samples.json")
for s in samples[:8]:
    tag = s["mode"] + ("" if s["sample_idx"] is None else f" #{s['sample_idx']}")
    display(Markdown(f"**{s['prompt']}** _({tag})_ → {s['continuation'][:400]}" + (" `[EOS]`" if s["ended_with_eos"] else "")))

In [ ]:
# attention pattern of the trained model (block 1, head 1) on a validation snippet -- lower-triangular by construction
from evaluate import load_model
best, _ = load_model(dirs["checkpoints"] / "best_model.pt", torch.device("cpu"))
snippet = "Once upon a time, there was a little girl named Lily. She loved to play."
ids = torch.tensor([encode(snippet, char_to_idx)[:best.cfg.block_size]])
best.blocks[0].attn.store_attn = True
with torch.no_grad():
    best(ids)
A = best.blocks[0].attn.last_attn[0, 0].numpy()
fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(A, cmap="viridis"); ax.set(title="Block 1 / head 1 attention", xlabel="key position", ylabel="query position")
plt.show()

## 5. Failure analysis (1.4)
`analyze_failures.py` ranks the generated samples by repetition, out-of-vocabulary (invented/broken) words,
coherence signals (character-name drift, never ending the story) and grammar signals. I read the
candidates and wrote up three real failure cases — snippet, failure type and observation — in
[`failure_analysis.md`](../failure_analysis.md).

In [ ]:
import subprocess
subprocess.run([sys.executable, str(SRC / "analyze_failures.py"), "--config", CONFIG, "--run-id", RUN_ID,
                "--set", *OVERRIDES], check=True)
display(Markdown((dirs["outputs"] / "failure_candidates.md").read_text(encoding="utf-8")[:6000]))

## 6. Evidence trail
* Raw logs (unedited): `reproducibility/raw_logs/task1_llm/shibin_thomas/<run_id>/` (`train.log`, `train_steps.csv`, `epochs.csv`, `eval.log`)
* Manifest (versions, hardware, git commit, config hash, checkpoint SHA-256): `reproducibility/manifests/task1_llm/shibin_thomas/<run_id>.json`
* Best checkpoint: `task1_llm/shibin_thomas/checkpoints/<run_id>/best_model.pt`
* Metrics: `task1_llm/shibin_thomas/metrics_report.csv` · write-up: `results.md`, `failure_analysis.md`

In [ ]:
print("run id:", RUN_ID)
print("manifest:", rel(dirs["manifests"] / f"{RUN_ID}.json"))
print(json.dumps({k: metrics[k] for k in ["val_cross_entropy", "val_perplexity", "val_bits_per_char",
                                         "val_top1_accuracy", "generalization_gap", "params_total"]}, indent=1))